# III-3 クラウド GPU を借りる：時間と費用を測って見積もり直す（ノートブック版）

教科書の III-3 のうち、**Python で計算する部分**（1 ステップの時間の記録、ウォームアップ・自己相関・疑似反復を考えた解析、費用の見積もり直し、中断と再開）を、上から順に実行すれば動く 1 冊にしたものです。Google Colab の GPU（CUDA）を、教科書の「借りたインスタンス」の代わりにします。Mac の Jupyter（MPS、または CPU）でも動きます。

| 節 | 内容 | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 解析の動きを合成データで確かめる（MSER-5、自己相関、疑似反復） | 6 | `cost_check.py --demo` |
| 2 | 借りる前の見積もり（III-1 の方法） | 6 | `bench_step.py`・`cost_estimator.py` |
| 3 | 1 ステップの時間を全部記録する | 12 | `bench_trace.py` |
| 4 | 本番の学習を中断して再開する（SIGTERM → チェックポイント） | 13・14 | `train_toy.py` |
| 5 | 記録から時間と費用を見積もり直す、データ待ちの比 η | 16 | `cost_check.py` |
| 6 | 発展課題1：マシンの差の分散成分と、借りる台数 | 発展1 | ― |
| 7 | 結果を持ち帰る | 15 | ― |

- 次の「準備」のセルたちが、III-1 の `bench_step.py`・`cost_estimator.py`、III-2 の `train_toy.py`、この章の `bench_trace.py`・`cost_check.py` を、このノートブックの隣に `%%writefile` で書き出します。**中身は教科書と 1 文字も変えていません**。
- 教科書のターミナルの `uv run python cost_check.py --demo` の代わりに、`run("cost_check.py --demo")` で同じ引数を渡して別のプロセスで実行します（`run()` はこのノートブックで足した小さな関数です）。図は PNG に保存されるので、セルの下に表示します。
- **既定の設定**：Colab の GPU（`cuda`）では教科書の本番と同じ（600 ステップ × 3 回、`--amp`）。Mac・CPU では小さい設定（200 ステップ × 3 回、学習は 3 エポック × 6000 個）です。本番の値はセルのコメントにあります。時間・費用の値は機械で変わるので、`ok()` は、合成データ（種が固定）の値と、記録の形・式の値など機械に依らないことだけを確かめます。
- 執筆時の Mac（Apple シリコン、MPS）で、全部で 3〜4 分ほどです。Colab の CPU のランタイム（GPU を選ばないとき）では、8〜10 分ほどかかる見込みです。
- **料金・為替は仮の値**です（教科書と同じ 0.40 USD/時、150 円/USD、60 円/時）。この教科書は料金を断定しません。自分で調べた値に書き換えてください。Colab の料金の数え方（コンピューティング ユニットなど）は Vast.ai と違うので、公式サイトで要確認です。
- **入れていないもの**：Vast.ai のアカウント・チャージ・鍵・インスタンスの作成と削除（手順 1〜5・7〜11・15。お金と SSH の操作で、ノートブックでは代わりになりません）、`vast_host.sh`・`vast_sync.sh`・`setup_instance.sh`・`vast_end_check.sh`（Mac とインスタンスのシェルで使うもの）、Git へのコミット。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

## 0. 準備

### Colab で GPU を選ぶ

1. メニューの「ランタイム」→「ランタイムのタイプを変更」を開き、ハードウェア アクセラレータで GPU（例：T4）を選んで保存します（項目の名前と、無料で選べる GPU の種類は公式ドキュメントで要確認）。
2. ランタイムを変えると、それまでに書き出したファイルと変数は消えるので、このノートブックは**最初のセルから**実行し直してください。
3. GPU を選ばなくても全部のセルは動きます（計算の場所が `cpu` になり、小さい設定になります）。

計算の場所（device）は、III-1・III-2 の `pick_device()` が **cuda → mps → cpu** の順に自動で選びます。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には torch・numpy・pandas・scipy・matplotlib・statsmodels が最初から入っている。版は公式で要確認）
import importlib, subprocess, sys
for mod, pkg in [("torch", "torch"), ("pandas", "pandas"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("statsmodels", "statsmodels")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, torch, platform
print("Python", platform.python_version(), " numpy", numpy.__version__, " torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), " MPS:", torch.backends.mps.is_available())

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

`bench_step.py`（III-1 の 6 節）を書き出します。

In [ ]:
%%writefile bench_step.py
#!/usr/bin/env python3
"""III-1：学習1ステップの所要時間とピークメモリを測る（Mac の MPS／CPU、クラウドの CUDA で共通）
使い方:
  uv run python bench_step.py --model unet --batch 8 --size 256 --out bench_mac_unet.json
  uv run python bench_step.py --model audio --batch 128 --size 4096 --out bench_mac_audio.json
  python bench_step.py --model unet --batch 8 --size 256 --out bench_gpu_unet.json   # クラウドで
"""
import argparse
import contextlib
import json
import math
import platform
import statistics
import time

import torch
import torch.nn as nn
import torch.nn.functional as F


def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def sync(dev):
    """GPU は非同期に動くので、時間を測る前後で計算の完了を待つ"""
    if dev.type == "cuda":
        torch.cuda.synchronize()
    elif dev.type == "mps":
        torch.mps.synchronize()


def device_name(dev):
    if dev.type == "cuda":
        return torch.cuda.get_device_name(0)
    if dev.type == "mps":
        return "Apple MPS (" + platform.machine() + ")"
    return platform.processor() or platform.machine()


def peak_mem_mb(dev):
    try:
        if dev.type == "cuda":
            return torch.cuda.max_memory_allocated() / 2**20
        if dev.type == "mps":
            return torch.mps.driver_allocated_memory() / 2**20   # キャッシュも含む目安
    except Exception:
        pass
    return None


class Block(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)


class TinyUNet(nn.Module):
    """3段の小さな U-Net（入力画像の一辺は 8 の倍数）"""
    def __init__(self, cin=1, ncls=2, w=32):
        super().__init__()
        self.e1, self.e2, self.e3 = Block(cin, w), Block(w, 2 * w), Block(2 * w, 4 * w)
        self.mid = Block(4 * w, 8 * w)
        self.u3, self.d3 = nn.ConvTranspose2d(8 * w, 4 * w, 2, stride=2), Block(8 * w, 4 * w)
        self.u2, self.d2 = nn.ConvTranspose2d(4 * w, 2 * w, 2, stride=2), Block(4 * w, 2 * w)
        self.u1, self.d1 = nn.ConvTranspose2d(2 * w, w, 2, stride=2), Block(2 * w, w)
        self.out = nn.Conv2d(w, ncls, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        m = self.mid(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(m), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)


class AudioCNN(nn.Module):
    """波形（1ch）を直接入れる小さな1次元CNN（2クラス分類）"""
    def __init__(self, w=32, ncls=2):
        super().__init__()
        layers, cin = [], 1
        for cout in (w, w, 2 * w, 2 * w, 4 * w, 4 * w):
            layers += [nn.Conv1d(cin, cout, 9, stride=2, padding=4),
                       nn.BatchNorm1d(cout), nn.ReLU(inplace=True)]
            cin = cout
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(cin, ncls)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=-1))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["unet", "audio"], default="unet")
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--size", type=int, default=256, help="unet: 画像の一辺、audio: 波形の長さ")
    ap.add_argument("--width", type=int, default=32)
    ap.add_argument("--warmup", type=int, default=10)
    ap.add_argument("--steps", type=int, default=50)
    ap.add_argument("--amp", action="store_true", help="CUDA で混合精度を使う")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--out", default="bench.json")
    a = ap.parse_args()
    if a.steps < 30:
        print("注意：--steps が 30 未満だと、区間の計算（正規近似）が粗くなります")

    dev = pick_device(a.device)
    torch.manual_seed(0)
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.cuda.reset_peak_memory_stats()
    if a.model == "unet":
        model = TinyUNet(w=a.width).to(dev)
        x = torch.randn(a.batch, 1, a.size, a.size, device=dev)
        y = torch.randint(0, 2, (a.batch, a.size, a.size), device=dev)
    else:
        model = AudioCNN(w=a.width).to(dev)
        x = torch.randn(a.batch, 1, a.size, device=dev)
        y = torch.randint(0, 2, (a.batch,), device=dev)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    use_amp = a.amp and dev.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    times = []
    for i in range(a.warmup + a.steps):
        sync(dev)
        t0 = time.perf_counter()
        opt.zero_grad(set_to_none=True)
        ctx = torch.autocast("cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()
        with ctx:
            loss = F.cross_entropy(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sync(dev)
        if i >= a.warmup:
            times.append(time.perf_counter() - t0)

    n = len(times)
    m = statistics.mean(times)
    sd = statistics.stdev(times)
    half = 1.96 * sd / math.sqrt(n)
    first, second = times[: n // 2], times[n // 2:]
    res = {
        "device": dev.type, "device_name": device_name(dev), "torch": torch.__version__,
        "model": a.model, "batch": a.batch, "size": a.size, "width": a.width, "amp": use_amp,
        "params": n_params, "warmup": a.warmup, "steps": n,
        "step_mean_s": m, "step_sd_s": sd, "step_ci95_s": [m - half, m + half],
        "step_mean_first_half_s": statistics.mean(first),
        "step_mean_second_half_s": statistics.mean(second),
        "samples_per_s": a.batch / m, "peak_mem_mb": peak_mem_mb(dev),
        "time": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    }
    with open(a.out, "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False, indent=2)
    print(f"{res['device_name']}  {a.model} batch={a.batch} size={a.size}  パラメータ {n_params/1e6:.2f}M")
    print(f"1ステップ = {m*1e3:.1f} ms（95%区間 {(m-half)*1e3:.1f}〜{(m+half)*1e3:.1f} ms、n={n}）")
    print(f"前半 {res['step_mean_first_half_s']*1e3:.1f} ms ／ 後半 {res['step_mean_second_half_s']*1e3:.1f} ms"
          "（後半が遅ければ発熱による速度低下を疑う）")
    pm = res["peak_mem_mb"]
    mem = f"ピークメモリ ≈ {pm:.0f} MB" if pm is not None else "ピークメモリ：（CPU では測らない）"
    print(f"処理量 = {res['samples_per_s']:.1f} サンプル/秒、{mem}")
    print(f"保存: {a.out}")


if __name__ == "__main__":
    main()

`cost_estimator.py`（III-1 の 6 節）を書き出します。

In [ ]:
%%writefile cost_estimator.py
#!/usr/bin/env python3
"""III-1：ベンチマーク結果から、クラウドGPUでの所要時間と費用を「区間つき」で見積もる
料金は自分で公式サイトを見て入力する（この教科書は料金を断定しない）。
使い方の例:
  # GPU で 10 分だけ測った結果がある場合
  uv run python cost_estimator.py --bench bench_gpu_unet.json --steps 30000 --price-jpy-h 60 --budget-jpy 1000
  # Mac の結果しかない場合（GPU での速度倍率を「5〜20倍のどこか」と幅で与える）
  uv run python cost_estimator.py --bench bench_mac_unet.json --speedup 5 20 --steps 30000 --price-jpy-h 60 --budget-jpy 1000
"""
import argparse
import json
import math

import numpy as np


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bench", required=True, help="bench_step.py の出力 JSON")
    ap.add_argument("--steps", type=int, required=True, help="本番の総ステップ数（エポック数 × 1エポックのステップ数）")
    ap.add_argument("--speedup", type=float, nargs=2, default=None, metavar=("LO", "HI"),
                    help="Mac の結果から推定するとき、GPU での速度倍率が 90%% の確からしさで入る範囲")
    ap.add_argument("--price-jpy-h", type=float, required=True, help="GPU の時間単価 [円/時]（公式サイトで確認した値）")
    ap.add_argument("--setup-min", type=float, default=20, help="環境構築・データ転送などの時間 [分]")
    ap.add_argument("--idle-min", type=float, default=15, help="監視・後片付けなど、計算していない時間 [分]")
    ap.add_argument("--disk-gb", type=float, default=0, help="借りるストレージの容量 [GB]")
    ap.add_argument("--disk-jpy-gb-month", type=float, default=0, help="ストレージ単価 [円/GB/月]")
    ap.add_argument("--keep-days", type=float, default=0, help="ストレージを残しておく日数")
    ap.add_argument("--egress-gb", type=float, default=0, help="転送量 [GB]")
    ap.add_argument("--egress-jpy-gb", type=float, default=0, help="転送単価 [円/GB]")
    ap.add_argument("--budget-jpy", type=float, default=None, help="今回の予算 [円]")
    ap.add_argument("--n-sim", type=int, default=100_000)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    with open(a.bench, encoding="utf-8") as f:
        b = json.load(f)
    rng = np.random.default_rng(a.seed)
    # 平均ステップ時間の不確かさ（中心極限定理による正規近似）
    se = b["step_sd_s"] / math.sqrt(b["steps"])
    step = np.clip(rng.normal(b["step_mean_s"], se, a.n_sim), 1e-9, None)
    if a.speedup:
        lo, hi = a.speedup
        mu = (math.log(lo) + math.log(hi)) / 2
        sigma = (math.log(hi) - math.log(lo)) / (2 * 1.645)   # 90% 範囲 → 対数正規分布の σ
        step = step / np.exp(rng.normal(mu, sigma, a.n_sim))
    run_h = step * a.steps / 3600
    gpu_h = run_h + (a.setup_min + a.idle_min) / 60
    cost = (a.price_jpy_h * gpu_h
            + a.disk_gb * a.disk_jpy_gb_month * a.keep_days / 30
            + a.egress_gb * a.egress_jpy_gb)

    p5, p50, p95 = np.percentile(run_h, [5, 50, 95])
    c5, c50, c95 = np.percentile(cost, [5, 50, 95])
    print(f"ベンチマーク: {b['device_name']}（{b['model']}, batch={b['batch']}）")
    print(f"計算時間   : 中央値 {p50:.2f} 時間（90%区間 {p5:.2f}〜{p95:.2f} 時間）")
    print(f"費用       : 中央値 {c50:,.0f} 円（90%区間 {c5:,.0f}〜{c95:,.0f} 円）")
    print(f"  内訳の目安：GPU {a.price_jpy_h*np.median(gpu_h):,.0f} 円、"
          f"ストレージ {a.disk_gb*a.disk_jpy_gb_month*a.keep_days/30:,.0f} 円、転送 {a.egress_gb*a.egress_jpy_gb:,.0f} 円")
    if a.budget_jpy:
        print(f"予算 {a.budget_jpy:,.0f} 円を超える確率 ≈ {(cost > a.budget_jpy).mean()*100:.1f} %")
    print(f"参考：止め忘れ12時間の費用 = {a.price_jpy_h*12:,.0f} 円")
    print("※ 見積もりはステップ時間の測定誤差と速度倍率の幅だけを考えている。"
          "データ読み込みの遅れや中断は含まないので、実際との比を台帳で確かめる。")


if __name__ == "__main__":
    main()

`train_toy.py`（III-2 の 6 節）を書き出します。

In [ ]:
%%writefile train_toy.py
#!/usr/bin/env python3
"""III-2：どこでも動く練習用の学習スクリプト
合成した「モータ音」の正常／異常を、1次元CNNで分類する。
- device を自動で選ぶ（cuda → mps → cpu）。同じコードが Mac と Colab で動く
- 一定時間ごと＋エポックの終わりにチェックポイントを保存。切断されても途中から正確に再開できる
- CUDA では混合精度（AMP）を使う
- --max-hours を過ぎたら、保存して自分で終了する（課金の止め忘れ対策）
使い方:
  uv run python train_toy.py --quick --ckpt-dir runs/mac_quick              # Mac で動作確認
  python train_toy.py --ckpt-dir /content/drive/MyDrive/home-lab/runs/toy01    # Colab で本番
"""
import argparse
import contextlib
import csv
import json
import math
import os
import platform
import random
import signal
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

FS = 16000
stop_requested = False


def on_signal(signum, frame):
    global stop_requested
    stop_requested = True
    print(f"\nシグナル {signum} を受信：区切りのよいところで保存して終了します", flush=True)


# ---------------------------------------------------------------- 環境
def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def device_name(dev):
    if dev.type == "cuda":
        return torch.cuda.get_device_name(0)
    if dev.type == "mps":
        return "Apple MPS"
    return platform.processor() or platform.machine()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def git_commit():
    try:
        r = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                           capture_output=True, text=True, timeout=5)
        return r.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def amp_ctx(enabled):
    return torch.autocast("cuda", dtype=torch.float16) if enabled else contextlib.nullcontext()


# ---------------------------------------------------------------- データ（合成）
def make_signal(rng, length, anomalous):
    """基本波 80〜120 Hz と高調波＋雑音。異常は「周期的なクリック」か「高い周波数のうなり」"""
    t = np.arange(length) / FS
    f0 = rng.uniform(80, 120)
    x = np.zeros(length)
    for k in range(1, 6):
        x += np.sin(2 * np.pi * k * f0 * t + rng.uniform(0, 2 * np.pi)) / k
    x += rng.normal(0, 0.3, length)
    if anomalous:
        if rng.integers(2) == 0:
            period = int(FS / rng.uniform(15, 40))
            win = np.hanning(16) * rng.uniform(2, 4)
            for p in range(int(rng.integers(period)), length, period):
                seg = x[p:p + 16]
                seg += win[:len(seg)]
        else:
            fm = rng.uniform(1500, 3000)
            env = 1 + 0.5 * np.sin(2 * np.pi * rng.uniform(5, 20) * t)
            x += 0.6 * np.sin(2 * np.pi * fm * t) * env
    return (x / x.std()).astype(np.float32)


class SynthMotor(Dataset):
    """番号 i から決まる乱数で毎回同じ信号を作る（ファイルに保存しないのでディスクを使わない）"""
    def __init__(self, n, length, seed):
        self.n, self.length, self.seed = n, length, seed

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, i])
        y = i % 2                                 # 正常と異常を半分ずつ
        x = make_signal(rng, self.length, bool(y))
        return torch.from_numpy(x).unsqueeze(0), y


# ---------------------------------------------------------------- モデル
class AudioCNN(nn.Module):
    def __init__(self, w=32, ncls=2):
        super().__init__()
        layers, cin = [], 1
        for cout in (w, w, 2 * w, 2 * w, 4 * w, 4 * w):
            layers += [nn.Conv1d(cin, cout, 9, stride=2, padding=4),
                       nn.BatchNorm1d(cout), nn.ReLU(inplace=True)]
            cin = cout
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(cin, ncls)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=-1))


# ---------------------------------------------------------------- チェックポイント
def save_ckpt(path, state):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)                         # 書き込み途中で切れても壊れない


@torch.no_grad()
def evaluate(model, loader, dev, use_amp):
    model.eval()
    correct = n = 0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        with amp_ctx(use_amp):
            pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        n += len(y)
    model.train()
    return correct / n


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--n-train", type=int, default=20000)
    ap.add_argument("--n-val", type=int, default=2000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--ckpt-dir", default="runs/toy")
    ap.add_argument("--ckpt-every-min", type=float, default=5.0)
    ap.add_argument("--max-hours", type=float, default=3.0, help="この時間を過ぎたら保存して終了")
    ap.add_argument("--quick", action="store_true", help="小さい設定で動作確認")
    a = ap.parse_args()
    if a.quick:
        a.n_train, a.n_val, a.epochs = 4000, 400, 2
    return a


def main():
    a = parse()
    signal.signal(signal.SIGTERM, on_signal)
    signal.signal(signal.SIGINT, on_signal)
    set_seed(a.seed)
    dev = pick_device(a.device)
    use_amp = dev.type == "cuda" and not a.no_amp
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
    os.makedirs(a.ckpt_dir, exist_ok=True)
    last = os.path.join(a.ckpt_dir, "last.pt")
    metrics_path = os.path.join(a.ckpt_dir, "metrics.csv")
    t_start = time.time()

    train_ds = SynthMotor(a.n_train, a.length, seed=a.seed)
    val_ds = SynthMotor(a.n_val, a.length, seed=a.seed + 10_000)
    pin = dev.type == "cuda"
    val_loader = DataLoader(val_ds, batch_size=a.batch * 2, num_workers=a.workers, pin_memory=pin)

    model = AudioCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch, start_step = 0, 0
    run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
    resumed = False
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)   # 自分で作ったファイルなので可
        if ck.get("done"):
            print(f"この実行はすでに完了しています: {last}")
            return
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        if use_amp and ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        start_epoch, start_step, run = ck["epoch"], ck["step"], ck["run"]
        resumed = True
        print(f"チェックポイントから再開: epoch {start_epoch} step {start_step}", flush=True)

    # 実行の記録（再開のたびに追記）
    info_path = os.path.join(a.ckpt_dir, "run_info.json")
    info = json.load(open(info_path, encoding="utf-8")) if os.path.exists(info_path) else {"sessions": []}
    info["sessions"].append({
        "started": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "git_commit": git_commit(),
        "torch": torch.__version__, "python": sys.version.split()[0],
        "device": dev.type, "device_name": device_name(dev), "amp": use_amp,
        "args": vars(a), "resumed": resumed})
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)
    print(f"device = {device_name(dev)}（{dev.type}）, AMP = {use_amp}, commit = {git_commit()}", flush=True)

    def state(epoch, step, run_state, done=False):
        return {"model": model.state_dict(), "opt": opt.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "step": step,
                "run": run_state, "args": vars(a), "done": done,
                "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

    last_save = time.time()
    for epoch in range(start_epoch, a.epochs):
        first = epoch == start_epoch
        if not first:
            run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
        step = start_step if first else 0
        # エポックごとの並び順を「種＋エポック番号」で決める → 途中からでも同じ順番で再開できる
        g = torch.Generator().manual_seed(a.seed * 1000 + epoch)
        order = torch.randperm(len(train_ds), generator=g).tolist()
        loader = DataLoader(Subset(train_ds, order[step * a.batch:]), batch_size=a.batch,
                            shuffle=False, num_workers=a.workers, pin_memory=pin)
        model.train()
        t_seg = time.time()
        for x, y in loader:
            x = x.to(dev, non_blocking=True)
            y = y.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_ctx(use_amp):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run["loss_sum"] += loss.item() * len(y)
            run["n"] += len(y)
            step += 1
            over_time = time.time() - t_start > a.max_hours * 3600
            if stop_requested or over_time or time.time() - last_save > a.ckpt_every_min * 60:
                run["t"] += time.time() - t_seg
                t_seg = time.time()
                save_ckpt(last, state(epoch, step, run))
                last_save = time.time()
                print(f"  チェックポイント保存: epoch {epoch} step {step}", flush=True)
                if stop_requested or over_time:
                    why = "シグナル" if stop_requested else f"--max-hours {a.max_hours}"
                    print(f"{why} により終了します。同じコマンドで再開できます。", flush=True)
                    return
        run["t"] += time.time() - t_seg
        val_acc = evaluate(model, val_loader, dev, use_amp)
        row = {"epoch": epoch, "train_loss": run["loss_sum"] / max(run["n"], 1),
               "val_acc": val_acc, "epoch_time_s": run["t"],
               "samples_per_s": run["n"] / max(run["t"], 1e-9),
               "device": dev.type, "device_name": device_name(dev),
               "resumed": resumed and first, "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")}
        new = not os.path.exists(metrics_path)
        with open(metrics_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row.keys()))
            if new:
                w.writeheader()
            w.writerow(row)
        print(f"epoch {epoch}: loss {row['train_loss']:.4f}  val_acc {val_acc:.4f}  "
              f"{row['samples_per_s']:.0f} サンプル/秒", flush=True)
        save_ckpt(last, state(epoch + 1, 0, {"loss_sum": 0.0, "n": 0, "t": 0.0},
                              done=(epoch + 1 == a.epochs)))
        last_save = time.time()
    print("完了しました。", flush=True)


if __name__ == "__main__":     # DataLoader の並列読み込み（macOS の spawn）に必要
    main()

`bench_trace.py`（III-3 の 6 節）を書き出します。

In [ ]:
%%writefile bench_trace.py
#!/usr/bin/env python3
"""III-3：学習1ステップの時間を「1ステップずつ全部」記録する（ウォームアップも捨てずに残す）
bench_step.py（III-1）のモデルと関数をそのまま使う。bench_step.py は平均と区間だけを出すが、
こちらは時系列を CSV に残し、ウォームアップの除き方・自己相関・マシンの差を cost_check.py で調べる。
使い方（クラウドのインスタンスで。同じフォルダに bench_step.py があること）:
  python bench_trace.py --model audio --batch 128 --size 4096 --reps 3 --amp --machine m1
  python bench_trace.py --model unet --batch 8 --size 256 --reps 3 --amp --machine m1
train_toy.py と比べるときは、同じ精度で測る（train_toy.py は CUDA で AMP を使うので --amp を付ける）。
  uv run python bench_trace.py --model audio --batch 128 --size 4096 --reps 3 --machine mac   # Mac で
"""
import argparse
import contextlib
import csv
import json
import os
import socket
import subprocess
import time

import torch
import torch.nn.functional as F

from bench_step import AudioCNN, TinyUNet, device_name, peak_mem_mb, pick_device, sync


def gpu_info():
    """同じ GPU 名でも、電力の上限やドライバの版はマシンごとに違うことがある。あとで比べるために残す"""
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,driver_version,power.limit,clocks.max.sm,memory.total",
             "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
        return r.stdout.strip()
    except Exception:
        return ""


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["unet", "audio"], default="audio")
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--size", type=int, default=4096, help="unet: 画像の一辺、audio: 波形の長さ")
    ap.add_argument("--width", type=int, default=32)
    ap.add_argument("--steps", type=int, default=600, help="1回の実行で測るステップ数（ウォームアップを含む）")
    ap.add_argument("--reps", type=int, default=3, help="同じマシンでくり返す回数（モデルを作り直す）")
    ap.add_argument("--pause-s", type=float, default=5.0, help="くり返しの間の休み [秒]")
    ap.add_argument("--amp", action="store_true", help="CUDA で混合精度を使う")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--machine", default=socket.gethostname(),
                    help="マシンの名前（コンソールに出るマシン番号などを自分で付ける。例 m1）")
    ap.add_argument("--outdir", default="runs/bench_trace")
    a = ap.parse_args()

    dev = pick_device(a.device)
    use_amp = a.amp and dev.type == "cuda"
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
    os.makedirs(a.outdir, exist_ok=True)
    stamp = time.strftime("%Y%m%d-%H%M%S")
    label = f"{a.machine}_{a.model}_b{a.batch}_{stamp}"
    csv_path = os.path.join(a.outdir, label + ".csv")
    gname = device_name(dev)

    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["label", "machine", "gpu", "model", "batch", "rep", "step", "t_s"])
        for rep in range(a.reps):
            torch.manual_seed(rep)
            if a.model == "unet":
                model = TinyUNet(w=a.width).to(dev)
                x = torch.randn(a.batch, 1, a.size, a.size, device=dev)
                y = torch.randint(0, 2, (a.batch, a.size, a.size), device=dev)
            else:
                model = AudioCNN(w=a.width).to(dev)
                x = torch.randn(a.batch, 1, a.size, device=dev)
                y = torch.randint(0, 2, (a.batch,), device=dev)
            opt = torch.optim.Adam(model.parameters(), lr=1e-3)
            scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
            if dev.type == "cuda":
                torch.cuda.reset_peak_memory_stats()
            for step in range(a.steps):
                sync(dev)
                t0 = time.perf_counter()
                opt.zero_grad(set_to_none=True)
                ctx = torch.autocast("cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()
                with ctx:
                    loss = F.cross_entropy(model(x), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sync(dev)                       # GPU の計算が終わるのを待ってから時計を読む
                w.writerow([label, a.machine, gname, a.model, a.batch, rep, step,
                            f"{time.perf_counter() - t0:.6f}"])
            f.flush()
            pm = peak_mem_mb(dev)
            mem = f"ピークメモリ ≈ {pm:.0f} MB" if pm is not None else "ピークメモリは測れない（CPU）"
            print(f"rep {rep}: {a.steps} ステップ記録（{mem}）", flush=True)
            del model, opt, x, y
            if dev.type == "cuda":
                torch.cuda.empty_cache()
            if rep + 1 < a.reps:
                time.sleep(a.pause_s)

    meta = {"label": label, "machine": a.machine, "device": dev.type, "device_name": gname,
            "nvidia_smi": gpu_info(), "torch": torch.__version__, "cuda": torch.version.cuda,
            "args": vars(a), "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")}
    with open(os.path.join(a.outdir, label + ".json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f"保存: {csv_path}")
    print(f"GPU の情報: {meta['nvidia_smi'] or '（nvidia-smi なし）'}")


if __name__ == "__main__":
    main()

`cost_check.py`（III-3 の 6 節）を書き出します。

In [ ]:
%%writefile cost_check.py
#!/usr/bin/env python3
"""III-3：クラウドで記録した1ステップの時間（bench_trace.py）から、本番の時間と費用を区間つきで見積もり直し、
III-1 の見積もり（cost_estimator.py）と実際の費用（残高の差）を比べる。
- ウォームアップを MSER-5 で決めて除く
- 自己相関を考えた区間（バッチ平均法）と、考えない区間を並べて出す
- 独立な反復の単位は「マシン」。同じマシンの中のステップや実行を、別々の反復として数えない（疑似反復）
使い方:
  uv run python cost_check.py --demo                    # 合成データで動きを確かめる（runs/demo_trace/ に作る）
  uv run python cost_check.py runs/bench_trace/m*_audio_*.csv --total-steps 15640 \
      --price-usd-h 0.40 --usd-jpy 150 --setup-min 20 --idle-min 15 \
      --disk-gb 40 --storage-usd-gb-month 0.15 --exist-h 0.9 \
      --estimate-jpy 35 50 95 --actual-jpy 70 --train-metrics runs/vast_toy01/metrics.csv
ベンチ（bench_trace.py）は train_toy.py と同じ精度で測る（CUDA なら --amp）。そろえないと η にデータ待ち以外の差が混ざる。
料金・為替は自分で調べた値を入れる（この教科書は料金を断定しない）。
"""
import argparse
import glob
import os

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False


# ---------------------------------------------------------------- 合成データ（--demo）
def make_demo(outdir, seed=1):
    """同じ GPU 名の3台（m1〜m3）で3回ずつ、600ステップ。ウォームアップ・自己相関・外れ値・マシンの差を入れる"""
    rng = np.random.default_rng(seed)
    os.makedirs(outdir, exist_ok=True)
    files = []
    for machine, k in [("m1", 1.00), ("m2", 1.07), ("m3", 0.96)]:
        base = 0.030 * k                                    # 1ステップ約30ms（マシンごとに数%違う）
        rows = []
        for rep in range(3):
            shift = rng.normal(0, 0.006) * base               # 実行ごとの小さなずれ
            e, ts = 0.0, []
            for step in range(600):
                e = 0.6 * e + rng.normal(0, 0.03 * base * np.sqrt(1 - 0.6**2))   # AR(1)、ρ=0.6
                t = base + shift + e + base * 3.0 * np.exp(-step / 5)            # ウォームアップ
                if step == 0:
                    t += 0.8                                                      # 最初の1回（準備）
                if rng.random() < 0.01:
                    t *= rng.uniform(2, 4)                                        # まれな外れ値
                ts.append(t)
            for step, t in enumerate(ts):
                rows.append([f"{machine}_demo", machine, "RTX 4070（合成）", "audio", 128, rep, step, round(t, 6)])
        df = pd.DataFrame(rows, columns=["label", "machine", "gpu", "model", "batch", "rep", "step", "t_s"])
        path = os.path.join(outdir, f"{machine}_audio_b128_demo.csv")
        df.to_csv(path, index=False)
        files.append(path)
    # train_toy.py の metrics.csv の形（データの生成を CPU で待つので、GPU だけの速さの約半分）
    m = pd.DataFrame({"epoch": range(4),
                      "samples_per_s": 128 / 0.030 * 0.55 * np.exp(rng.normal(0, 0.03, 4)),
                      "device_name": "RTX 4070（合成）"})
    m.loc[0, "samples_per_s"] *= 0.8                          # 最初のエポックは遅い
    mpath = os.path.join(outdir, "metrics_demo.csv")
    m.to_csv(mpath, index=False)
    return files, mpath


# ---------------------------------------------------------------- 1回の実行の要約
def mser5(x):
    """MSER-5：5個ずつの平均の列で、残りの平均の標準誤差の2乗 ≈ var/(n-d) が最小になる d を選ぶ（後半は探さない）"""
    nb = len(x) // 5
    b = x[: nb * 5].reshape(nb, 5).mean(axis=1)
    best_d, best = 0, np.inf
    for d in range(nb // 2 + 1):
        rest = b[d:]
        v = rest.var() / len(rest)
        if v < best:
            best_d, best = d, v
    return best_d * 5


def acf(x, maxlag=20):
    x = x - x.mean()
    c0 = np.dot(x, x) / len(x)
    return np.array([np.dot(x[: len(x) - k], x[k:]) / len(x) / c0 for k in range(maxlag + 1)])


def batch_means_ci(x, k=10, conf=0.95):
    """k 個の塊に分けて塊の平均を独立とみなす。自己相関があっても区間が狭くなりすぎない"""
    m = len(x) // k
    means = x[len(x) - m * k:].reshape(k, m).mean(axis=1)
    half = stats.t.ppf((1 + conf) / 2, k - 1) * means.std(ddof=1) / np.sqrt(k)
    return means.mean(), half


def summarize_rep(x, warmup=None):
    d = mser5(x) if warmup is None else warmup
    y = x[d:]
    n = len(y)
    med = np.median(y)
    mad = 1.4826 * np.median(np.abs(y - med))
    out = int((y > med + 6 * mad).sum())
    r1 = acf(np.minimum(y, med + 6 * mad), 1)[1]       # 外れ値で自己相関が見えなくならないよう、上を切って計算
    n_eff = n * (1 - r1) / (1 + r1) if r1 > 0 else n
    naive_half = 1.96 * y.std(ddof=1) / np.sqrt(n)
    _, bm_half = batch_means_ci(y)
    return dict(warmup=d, n=n, mean=y.mean(), median=med, sd=y.std(ddof=1), rho1=r1, n_eff=n_eff,
                naive_half=naive_half, bm_half=bm_half, outliers=out)


# ---------------------------------------------------------------- 本体
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv", nargs="*", help="bench_trace.py の CSV（ワイルドカード可）")
    ap.add_argument("--demo", action="store_true", help="合成データを作って解析する")
    ap.add_argument("--demo-dir", default="runs/demo_trace")
    ap.add_argument("--warmup", type=int, default=None, help="除くステップ数を固定する（省略すると MSER-5 で決める）")
    ap.add_argument("--total-steps", type=int, default=15640, help="本番の総ステップ数（train_toy.py --epochs 20 --n-train 100000 なら 15640）")
    ap.add_argument("--price-usd-h", type=float, default=None, help="時間単価 [USD/時]（借りたときの画面の値）")
    ap.add_argument("--usd-jpy", type=float, default=None, help="為替 [円/USD]（調べた日の値）")
    ap.add_argument("--setup-min", type=float, default=20)
    ap.add_argument("--idle-min", type=float, default=15)
    ap.add_argument("--disk-gb", type=float, default=0)
    ap.add_argument("--storage-usd-gb-month", type=float, default=0)
    ap.add_argument("--exist-h", type=float, default=0, help="インスタンスが存在した時間 [時]（停止中も含む）")
    ap.add_argument("--estimate-jpy", type=float, nargs=3, default=None, metavar=("P5", "P50", "P95"),
                    help="III-1 の cost_estimator.py の 5%%点・中央値・95%%点")
    ap.add_argument("--actual-jpy", type=float, default=None, help="実際の費用（作る前と削除した後の残高の差を円にしたもの）")
    ap.add_argument("--train-metrics", default=None, help="train_toy.py の metrics.csv（データ待ちの割合を見る）")
    ap.add_argument("--out", default="cost_check.png")
    a = ap.parse_args()

    files = []
    for p in a.csv:
        files += sorted(glob.glob(p))
    if a.demo:
        files, a.train_metrics = make_demo(a.demo_dir)
        a.price_usd_h = a.price_usd_h or 0.40       # 仮の値
        a.usd_jpy = a.usd_jpy or 150.0              # 仮の値
        a.setup_min, a.idle_min = 20, 15
        a.disk_gb, a.storage_usd_gb_month, a.exist_h = 40, 0.15, 3   # 仮の値
        a.total_steps = 15640                       # train_toy.py --epochs 20 --n-train 100000（782 ステップ×20）
        a.estimate_jpy = a.estimate_jpy or [35, 50, 95]
        a.actual_jpy = a.actual_jpy or 70
        print("※ 合成データ（--demo）。料金・為替・見積もり・実際の費用はすべて仮の値です\n")
    if not files:
        ap.error("CSV がありません（--demo で試せます）")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    if df["model"].nunique() > 1 or df["batch"].nunique() > 1:
        ap.error("モデルとバッチサイズが同じ CSV だけを渡してください")
    if df["gpu"].nunique() > 1:
        print(f"⚠ GPU の違うマシンが混ざっています（{', '.join(df['gpu'].unique())}）。"
              "Mac の CSV を混ぜていないか確かめる。マシンの差の区間は、同じ GPU 名どうしで出すもの\n")

    # ---- 1回の実行ごと
    rows = []
    for (machine, label, rep), g in df.groupby(["machine", "label", "rep"]):
        s = summarize_rep(g.sort_values("step")["t_s"].to_numpy(), a.warmup)
        rows.append(dict(machine=machine, label=label, rep=rep, gpu=g["gpu"].iloc[0], **s))
    rs = pd.DataFrame(rows)
    how = "MSER-5 で決める" if a.warmup is None else f"--warmup {a.warmup} で固定"
    print(f"1回の実行ごと（除くウォームアップ：{how}。時間は ms）")
    print(" マシン rep 除外 n    平均   中央値  ρ1    n_eff  素朴な±  バッチ平均±  外れ値")
    for r in rs.itertuples():
        warn = "  ←切り捨てが多い" if r.warmup > (r.warmup + r.n) / 4 else ""
        print(f" {r.machine:5s} {r.rep:3d} {r.warmup:4d} {r.n:4d} {r.mean*1e3:7.2f} {r.median*1e3:7.2f}"
              f" {r.rho1:5.2f} {r.n_eff:6.0f} {r.naive_half*1e3:7.2f}  {r.bm_half*1e3:9.2f}  {r.outliers:5d}{warn}")
    if (rs.warmup > (rs.warmup + rs.n) / 4).any():
        print(" → 「切り捨てが多い」実行：MSER-5 は自己相関が強いと切り捨てすぎることがある。(a) の図で"
              "途中から速さが変わっていない（発熱・クロック）かを確かめ、--warmup 50 の結果と平均を比べる")
    print(" → ρ1 が正なら、素朴な区間（ステップを独立とみなす）は狭すぎる。バッチ平均法の区間を使う。")
    print(" → 外れ値は捨てない。費用は合計時間で決まるので、費用には平均を、典型的な速さには中央値を使う。\n")

    # ---- マシンの差（独立な反復の単位はマシン）
    mm = rs.groupby("machine")["mean"].agg(["mean", "count"])
    k = len(mm)
    grand = mm["mean"].mean()
    pooled = np.concatenate([g.sort_values("step")["t_s"].to_numpy()[int(rs[(rs.label == lab) & (rs.rep == rep)].warmup.iloc[0]):]
                             for (lab, rep), g in df.groupby(["label", "rep"])])
    naive_half = 1.96 * pooled.std(ddof=1) / np.sqrt(len(pooled))
    print(f"マシン {k} 台（GPU 名：{', '.join(rs['gpu'].unique())}）")
    if k >= 2:
        t = stats.t.ppf(0.975, k - 1)
        half = t * mm["mean"].std(ddof=1) / np.sqrt(k)
        # 一元配置の変量効果：実行の平均 = 全体 + マシンの効果 + 実行のずれ
        r_i = mm["count"].to_numpy()
        N = r_i.sum()
        msb = (r_i * (mm["mean"].to_numpy() - rs["mean"].mean()) ** 2).sum() / (k - 1)
        msw = ((rs["mean"] - rs["machine"].map(mm["mean"])) ** 2).sum() / max(N - k, 1)
        n0 = (N - (r_i**2).sum() / N) / (k - 1)
        sb2 = max(0.0, (msb - msw) / n0)
        icc = sb2 / (sb2 + msw) if sb2 + msw > 0 else 0.0
        f_p = stats.f.sf(msb / msw, k - 1, N - k) if msw > 0 else float("nan")
        print(f" マシンの平均の平均 = {grand*1e3:.2f} ms（95%区間 ±{half*1e3:.2f} ms、t 分布 自由度 {k-1}）")
        print(f" 全ステップを独立とみなした区間 = ±{naive_half*1e3:.3f} ms（n={len(pooled)}）← 疑似反復。"
              f"{half/naive_half:.0f} 倍も狭く、この GPU 全体についての区間ではない")
        print(f" マシン間の SD = {np.sqrt(sb2)*1e3:.2f} ms、実行間の SD = {np.sqrt(msw)*1e3:.2f} ms、"
              f"ICC = {icc:.2f}（F 検定 p = {f_p:.3g}）")
    else:
        half = rs["bm_half"].max()
        print(" マシンが1台だけ：区間は「このマシン」についてのもので、同じ GPU 名のほかのマシンには広げられない")
    print()

    # ---- データ待ちの割合（train_toy.py と比べる）
    slow = 1.0
    if a.train_metrics and os.path.exists(a.train_metrics):
        m = pd.read_csv(a.train_metrics)
        s = m["samples_per_s"].to_numpy()
        s = s[1:] if len(s) >= 3 else s                      # compare_runs.py と同じく最初のエポックを除く
        train_rate = float(np.exp(np.log(s).mean()))
        bench_rate = df["batch"].iloc[0] / grand
        print(f"train_toy.py の処理量 {train_rate:,.0f} サンプル/秒 ÷ bench の処理量 {bench_rate:,.0f} サンプル/秒"
              f" = {train_rate/bench_rate:.2f}")
        print(" → 1 より小さい分は、GPU がデータ（CPU での生成・読み込み）を待っている時間の目安（III-5 で切り分ける）")
        print("   ※ ベンチを train_toy.py と同じ精度（CUDA なら --amp）で測ったときだけ、こう読める\n")
        slow = bench_rate / train_rate

    # ---- 本番の時間と費用
    lo, hi = grand - half, grand + half
    T = np.array([lo, grand, hi]) * a.total_steps / 3600
    print(f"本番 {a.total_steps} ステップの計算時間 ≈ {T[1]:.2f} 時間（{T[0]:.2f}〜{T[2]:.2f}）※ bench は GPU だけの速さ")
    proj = None
    if a.price_usd_h and a.usd_jpy:
        p_jpy = a.price_usd_h * a.usd_jpy
        storage = a.disk_gb * a.storage_usd_gb_month * a.usd_jpy * a.exist_h / (30 * 24)
        over = (a.setup_min + a.idle_min) / 60
        proj_gpu = p_jpy * (T + over) + storage
        proj = p_jpy * (T * slow + over) + storage
        print(f"時間単価 {a.price_usd_h} USD/時 × {a.usd_jpy} 円/USD = {p_jpy:.0f} 円/時（仮の値なら必ず自分の値に）")
        print(f"見積もり直した費用（GPU だけの速さ）  ≈ {proj_gpu[1]:,.0f} 円（{proj_gpu[0]:,.0f}〜{proj_gpu[2]:,.0f} 円）")
        if slow != 1.0:
            print(f"見積もり直した費用（データ待ちを含む）≈ {proj[1]:,.0f} 円（{proj[0]:,.0f}〜{proj[2]:,.0f} 円）"
                  f"  ← train_toy.py の処理量で {slow:.2f} 倍した")
        print(f"  うちストレージ {storage:,.0f} 円、準備と待ち {p_jpy*over:,.0f} 円（中央値の {p_jpy*over/proj[1]*100:.0f}%）")
    if a.estimate_jpy:
        e5, e50, e95 = a.estimate_jpy
        print(f"III-1 の見積もり（cost_estimator.py）: 中央値 {e50:,.0f} 円（90%区間 {e5:,.0f}〜{e95:,.0f} 円）")
    if a.actual_jpy is not None:
        print(f"実際の費用（残高の差）: {a.actual_jpy:,.0f} 円")
        if a.estimate_jpy:
            r = a.actual_jpy / a.estimate_jpy[1]
            inside = a.estimate_jpy[0] <= a.actual_jpy <= a.estimate_jpy[2]
            print(f" 実際 ÷ 見積もり = {r:.2f}（ln r = {np.log(r):+.2f}）。見積もりの 90% 区間の"
                  f"{'中' if inside else '外'}")
            print(" → cost_ledger.csv の estimate_jpy に見積もりの中央値、actual_jpy に実際の値を書く"
                  "（ledger_summary.py が比の区間を出す）")
        if proj is not None and a.actual_jpy > proj[2]:
            print(" → 見積もり直した区間より高い：準備・待ち時間（setup・idle）、データ待ち、ストレージ、通信を見直す")

    # ---- 図
    fig, ax = plt.subplots(2, 2, figsize=(11, 8))
    g0 = df[(df.label == rs.label.iloc[0]) & (df.rep == rs.rep.iloc[0])].sort_values("step")
    x0 = g0["t_s"].to_numpy() * 1e3
    d0 = int(rs.warmup.iloc[0])
    ax[0, 0].plot(g0["step"], x0, lw=0.8)
    ax[0, 0].axvline(d0, color="C3", ls="--", label=f"MSER-5 の切り捨て点（{d0}）")
    ax[0, 0].set(xlabel="ステップ", ylabel="1ステップの時間 [ms]", yscale="log",
                 title=f"(a) {rs.machine.iloc[0]} rep {rs.rep.iloc[0]} の時系列")
    ax[0, 0].legend()
    y0 = x0[d0:]
    m0 = np.median(y0)
    y0 = np.minimum(y0, m0 + 6 * 1.4826 * np.median(np.abs(y0 - m0)))
    r = acf(y0, 20)
    ax[0, 1].bar(range(len(r)), r)
    ax[0, 1].axhspan(-1.96 / np.sqrt(len(y0)), 1.96 / np.sqrt(len(y0)), color="0.85")
    ax[0, 1].set(xlabel="ずれ（ステップ）", ylabel="自己相関", title="(b) ウォームアップ後の自己相関（外れ値の上を切る）")
    machines = list(mm.index)
    for i, mname in enumerate(machines):
        v = rs[rs.machine == mname]["mean"] * 1e3
        ax[1, 0].plot(np.full(len(v), i), v, "o", color="0.5")
        ax[1, 0].plot(i, v.mean(), "D", color="C0", ms=9)
    ax[1, 0].errorbar(len(machines) + 0.3, grand * 1e3, yerr=naive_half * 1e3, fmt="s", color="C3",
                      capsize=6, label="全ステップを独立とみなす（誤り）")
    ax[1, 0].errorbar(len(machines) + 0.7, grand * 1e3, yerr=half * 1e3, fmt="s", color="C2",
                      capsize=6, label="マシンを単位にした区間")
    ax[1, 0].set_xticks(list(range(len(machines))) + [len(machines) + 0.5], machines + ["全体"])
    ax[1, 0].set(ylabel="実行ごとの平均 [ms]", title="(c) マシンの差と疑似反復")
    ax[1, 0].legend(fontsize=8)
    yl, labels = 0, []
    if a.estimate_jpy:
        ax[1, 1].plot([a.estimate_jpy[0], a.estimate_jpy[2]], [yl, yl], lw=6, color="C1", alpha=0.6)
        ax[1, 1].plot(a.estimate_jpy[1], yl, "o", color="C1")
        labels.append("III-1 の見積もり")
        yl += 1
    if proj is not None:
        ax[1, 1].plot([proj[0], proj[2]], [yl, yl], lw=6, color="C2", alpha=0.6)
        ax[1, 1].plot(proj[1], yl, "o", color="C2")
        labels.append("クラウドで測って見積もり直し")
        yl += 1
    if a.actual_jpy is not None:
        ax[1, 1].axvline(a.actual_jpy, color="C3", ls="--", label="実際（残高の差）")
        ax[1, 1].legend()
    ax[1, 1].set_yticks(range(len(labels)), labels)
    ax[1, 1].set_ylim(-0.7, max(len(labels), 1) - 0.3)
    ax[1, 1].set(xlabel="費用 [円]", title="(d) 費用：見積もりと実際")
    fig.tight_layout()
    fig.savefig(a.out, dpi=150)
    print(f"\n{a.out} を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
# 設定と、run()・ok()・show() を用意する（ノートブック版で足したもの）
import os, re, sys, json, glob, shlex, shutil, signal, subprocess, time
import numpy as np
import pandas as pd
from IPython.display import Image, display
sys.path.insert(0, ".")
from bench_step import pick_device, device_name

IN_COLAB = "google.colab" in sys.modules
DEV = pick_device()
CUDA = DEV.type == "cuda"
MACHINE = ("colab" if IN_COLAB else "local") if CUDA else "mac" if DEV.type == "mps" else "cpu"   # 教科書の --machine m1 の代わり
AMP = "--amp" if CUDA else ""          # train_toy.py は CUDA で AMP を使うので、ベンチも同じ精度で測る（III-3-6）
STEPS = 600 if CUDA else 200          # bench_trace.py の 1 回のステップ数（教科書の本番は 600）
REPS, PAUSE = 3, (5 if CUDA else 1)   # くり返し 3 回、休み（教科書は 5 秒）
TOY = "--epochs 3 --n-train 6000 --n-val 500 --workers 2"   # 小さい設定
# 本番（教科書の手順12）：TOY = "--epochs 20 --n-train 100000 --workers 8"（15640 ステップ。Colab の T4 では 1 時間前後かかることがある）
PRICE_USD_H, USD_JPY = 0.40, 150      # 仮の値。自分で調べた値に（この教科書は料金を断定しない）
os.makedirs("runs", exist_ok=True)
print(f"device {DEV}（{device_name(DEV)}）、machine {MACHINE}、bench {STEPS} ステップ×{REPS} 回 {AMP or '（AMP なし）'}")


def run(cmd, check=True):
    """ターミナルの `uv run python <cmd>` の代わり。別のプロセスで実行し、出力をそのまま表示して返す"""
    args = [sys.executable, "-u", *shlex.split(cmd)]
    print("$ python", cmd, flush=True)
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    rc = p.wait()
    if check:
        assert rc == 0, f"エラーで終わりました（returncode {rc}）。上の表示を見てください"
    return "".join(lines)


def num(pattern, text):
    """表示の中から数値を1つ取り出す（ok() で使う）"""
    m = re.search(pattern, text)
    assert m, "表示に見つかりません: " + pattern
    return float(m.group(1).replace(",", ""))


N_OK = 0
def ok(cond, msg):
    global N_OK
    assert cond, "★確かめ失敗：" + msg
    N_OK += 1
    print("OK", msg)


def show(png):
    display(Image(filename=png, width=900))

## 1. 解析の動きを合成データで確かめる（手順 6、`cost_check.py --demo`）

借りる前に、解析の道具を合成データで試します。同じ GPU 名の 3 台（m1〜m3）で 3 回ずつ 600 ステップを作り、ウォームアップ（最初の数ステップが遅い）、自己相関（AR(1)、$\rho = 0.6$）、まれな外れ値、マシンの差（数%）を入れてあります。

- **MSER-5**：5 個ずつの平均の列で、残りの平均の標準誤差が最小になる切り捨て点 $d$ を選ぶ（III-3-3）
- **バッチ平均法**：塊の平均を独立とみなすので、自己相関があっても区間が狭くなりすぎない
- **疑似反復**：独立な反復の単位は「マシン」。全ステップを独立とみなした区間は、この GPU の種類についての区間ではない

料金・見積もり・実際の費用は、`--demo` が入れる仮の値です。

In [ ]:
out1 = run("cost_check.py --demo")
show("cost_check.png")

In [ ]:
# 確かめ（教科書 III-3-5 の「確かめること」と III-3-7 の結果の例のうち、機械に依らず成り立つことだけ）
from cost_check import make_demo, summarize_rep, mser5
files, _ = make_demo("runs/demo_check")
df = pd.concat([pd.read_csv(f) for f in files])
rs = pd.DataFrame([dict(machine=m, rep=r, **summarize_rep(g.sort_values("step")["t_s"].to_numpy())) for (m, r), g in df.groupby(["machine", "rep"])])
ok(len(df) == 3 * 3 * 600, "[手順6] 合成データは 3 台 × 3 回 × 600 ステップ")
ok(15 <= rs.warmup.median() <= 30 and (rs.warmup >= 100).any() and "切り捨てが多い" in out1,
   f"III-3-7：切り捨ては多くの実行で 15〜30 ステップ（中央値 {rs.warmup.median():.0f}）、100 以上の実行もあり「切り捨てが多い」と出る")
ok(abs(rs.rho1.mean() - 0.5) < 0.07, f"III-3-7：ρ1 は約 0.5（平均 {rs.rho1.mean():.2f}。真の値は 0.6 だが、外れ値・ウォームアップの残りで少し小さく出る）")
ok(abs((rs.n_eff / rs.n).mean() - 1 / 3) < 0.07, f"III-3-7：n_eff は n の約 3 分の 1（{(rs.n_eff / rs.n).mean():.2f}）。(1−ρ)/(1+ρ) = 1/3（ρ = 0.5）")
grand, half, naive = num(r"平均の平均 = ([\d.]+) ms", out1), num(r"95%区間 ±([\d.]+) ms", out1), num(r"独立とみなした区間 = ±([\d.]+) ms", out1)
ok(0.10 <= half / grand <= 0.20, f"III-3-7：マシン単位の 95%区間（自由度 2）は ±{half / grand * 100:.0f}%（±十数%）")
ok(20 <= half / naive <= 31, f"III-3-7：全ステップを独立とみなした区間は、その {half / naive:.0f} 分の 1（20〜30 分の 1）")
ok(abs(num(r"サンプル/秒 = ([\d.]+)", out1) - 0.57) < 0.015, "III-3-7：データ待ちの比 η は約 0.57")
ok(os.path.exists("cost_check.png"), "[手順6] cost_check.png ができる")
# MSER-5 の動き：ウォームアップが 40 ステップの、自己相関の無い列なら、切り捨て点はその近く
rng = np.random.default_rng(0)
x = 0.03 + rng.normal(0, 0.0005, 600); x[:40] += 0.02
ok(35 <= mser5(x) <= 50, f"MSER-5 は、はっきりしたウォームアップ（40 ステップ）を {mser5(x)} ステップで切る")

## 2. 借りる前の見積もり（手順 6、III-1 の `bench_step.py`・`cost_estimator.py`）

III-1 の方法で、本番（`train_toy.py --epochs 20 --n-train 100000`、782 ステップ × 20 = 15640 ステップ）の費用を見積もります。

- Mac（MPS・CPU）で動かしているときは、教科書と同じく「GPU での速度倍率は 5〜20 倍のどこか」（`--speedup 5 20`）と幅で与えます。
- Colab の GPU で動かしているときは、ベンチそのものが GPU の速さなので `--speedup` を付けません（教科書の (2) の `bench_gpu_audio_m1.json` を渡す場合と同じ）。

時間単価 60 円/時（0.40 USD × 150 円）は仮の値です。5%点・中央値・95%点をメモし、5 節で `--estimate-jpy` に渡します。

In [ ]:
run(f"bench_step.py --model audio --batch 128 --size 4096 --steps 200 {AMP} --out runs/bench_{MACHINE}_audio.json")
SPEED = "" if CUDA else "--speedup 5 20"
out2 = run(f"cost_estimator.py --bench runs/bench_{MACHINE}_audio.json {SPEED} --steps 15640 "
           f"--price-jpy-h {PRICE_USD_H * USD_JPY:.0f} --setup-min 20 --idle-min 15 --budget-jpy 300")
m = re.search(r"費用\s*: 中央値 ([\d,]+) 円（90%区間 ([\d,]+)〜([\d,]+) 円）", out2)
EST = [float(v.replace(",", "")) for v in (m.group(2), m.group(1), m.group(3))]
print("見積もりの 5%点・中央値・95%点 =", EST, "円 → 5 節の --estimate-jpy に渡す")

In [ ]:
# 確かめ（教科書 III-3-5 の「確かめること」と III-3-7 の結果の例のうち、機械に依らず成り立つことだけ）
b = json.load(open(f"runs/bench_{MACHINE}_audio.json"))
gm = 1.0 if CUDA else (5 * 20) ** 0.5          # 対数正規分布の中央値＝ 5 と 20 の幾何平均 10
expect = PRICE_USD_H * USD_JPY * (b["step_mean_s"] / gm * 15640 / 3600 + 35 / 60)
ok(abs(EST[1] / expect - 1) < 0.03, f"cost_estimator.py の費用の中央値 {EST[1]:.0f} 円 ≈ 時間単価 ×（1 ステップ ÷ {gm:.0f} × 15640 ステップ ＋ 準備と待ち 35 分）= {expect:.0f} 円")
ok(EST[0] <= EST[1] <= EST[2], "[手順6] 見積もりの 3 つの値（5%点 ≤ 中央値 ≤ 95%点）がメモできた")
ok(abs(PRICE_USD_H * USD_JPY * 35 / 60 - 35) < 1e-9, "準備と待ち（20 分＋15 分）だけで 60 円/時 × 35/60 時間 = 35 円（本番の計算が短いと、ここが費用の大半になる。III-3-7）")

## 3. 1 ステップの時間を全部記録する（手順 12 の (1)、`bench_trace.py`）

`bench_step.py` は平均と区間だけを出しますが、`bench_trace.py` はウォームアップも捨てずに、全部のステップの時間を CSV に残します。GPU の計算は非同期なので、時計を読む前に `sync()` で計算が終わるのを待っています。

教科書ではインスタンスの tmux の中で `python bench_trace.py --model audio --batch 128 --size 4096 --reps 3 --amp --machine m1 --outdir /workspace/runs/bench_trace` を実行します。ここでは `--machine` に `colab`（Mac なら `mac`）を付けます。

In [ ]:
shutil.rmtree("runs/bench_trace", ignore_errors=True)   # やり直したときに前の CSV が混ざらないように
run(f"bench_trace.py --model audio --batch 128 --size 4096 --steps {STEPS} --reps {REPS} --pause-s {PAUSE} {AMP} "
    f"--machine {MACHINE} --outdir runs/bench_trace")
# 本番（教科書の (1)。インスタンス）：--steps 600（既定）--reps 3 --amp --machine m1

In [ ]:
# 確かめ（教科書 III-3-5 の「確かめること」と III-3-7 の結果の例のうち、機械に依らず成り立つことだけ）
csvs = sorted(glob.glob(f"runs/bench_trace/{MACHINE}_audio_*.csv"))
tr = pd.read_csv(csvs[-1])
meta = json.load(open(csvs[-1][:-4] + ".json"))
ok(len(tr) == STEPS * REPS and list(tr.columns) == ["label", "machine", "gpu", "model", "batch", "rep", "step", "t_s"],
   f"[手順12] CSV は {STEPS} × {REPS} = {STEPS * REPS} 行＋見出し（本番は 1800 行）")
ok(meta["device"] == DEV.type and meta["args"]["amp"] == CUDA, "JSON に device・引数（--amp）が残る")
if CUDA and shutil.which("nvidia-smi"):
    ok(meta["nvidia_smi"] != "", "[手順12] JSON の nvidia_smi に GPU 名・ドライバ・電力の上限などが記録されている：" + meta["nvidia_smi"])
first = tr[tr.step == 0].t_s.mean(); later = tr[tr.step >= 50].t_s.median()
print(f"最初の 1 ステップ {first * 1e3:.0f} ms、50 ステップ目以降の中央値 {later * 1e3:.1f} ms"
      "（III-3-7：最初の 1 ステップは数百 ms〜1 秒以上かかり、その後数ステップで落ち着く。Mac では差が小さいことがある）")

## 4. 本番の学習を中断して再開する（手順 13・14、III-2 の `train_toy.py`）

教科書では、インスタンスの tmux の中で `train_toy.py` を回し、別のウィンドウから `pkill -TERM -o -f "python train_toy.py"` で**いちばん古い（親の）プロセスだけ**に SIGTERM を送ります。`train_toy.py` は「シグナル 15 を受信」→ 区切りのよいステップでチェックポイントを保存して終了し、同じコマンドで「チェックポイントから再開」します。

ここでは、`subprocess.Popen` で学習を裏で始め、最初のエポックが終わったところで、親のプロセス（`p.pid`）にだけ `SIGTERM` を送ります（`pkill -o` と同じ。DataLoader のワーカーには送りません）。そのあと同じ引数でもう一度実行します。

**既定は小さい設定**（3 エポック × 6000 個、ワーカー 2）です。本番は 0 節の `TOY` のコメントのとおりです。

In [ ]:
CK = "runs/colab_toy01" if IN_COLAB else f"runs/{MACHINE}_toy01"
shutil.rmtree(CK, ignore_errors=True)
args = [sys.executable, "-u", "train_toy.py", *shlex.split(TOY), "--ckpt-dir", CK, "--ckpt-every-min", "5", "--max-hours", "1"]
print("$", " ".join(args[1:]))
p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
log1 = []
for line in p.stdout:
    print(line, end="", flush=True)
    log1.append(line)
    if line.startswith("epoch 0:"):
        time.sleep(0.5)                      # 2 エポック目の途中で
        print(f"--- 親のプロセス（PID {p.pid}）にだけ SIGTERM を送る（pkill -TERM -o と同じ）---", flush=True)
        p.send_signal(signal.SIGTERM)
rc1 = p.wait()
print("終了コード", rc1)
log2 = run(" ".join(args[2:]))               # 同じコマンドでもう一度 → 再開

In [ ]:
# 確かめ（教科書 III-3-5 の「確かめること」と III-3-7 の結果の例のうち、機械に依らず成り立つことだけ）
log1 = "".join(log1)
ok("シグナル 15 を受信" in log1 and "チェックポイント保存" in log1 and "同じコマンドで再開できます" in log1,
   "[手順14] 「シグナル 15 を受信」→「チェックポイント保存」→ 終了")
ok("チェックポイントから再開: epoch 1" in log2 and "完了しました" in log2, "[手順14] 再実行で「チェックポイントから再開: epoch 1 step …」、最後に「完了しました」")
mt = pd.read_csv(f"{CK}/metrics.csv")
ok(len(mt) == 3 and mt.resumed.astype(str).eq("True").sum() == 1 and str(mt.resumed.iloc[1]) == "True",
   "[手順14] metrics.csv は 3 エポック分で、再開したエポック（1）の resumed が True")
ok(len(json.load(open(f"{CK}/run_info.json"))["sessions"]) == 2, "run_info.json に 2 回の実行（最初と再開）が追記されている")
print(mt[["epoch", "train_loss", "val_acc", "samples_per_s", "resumed"]])

## 5. 記録から時間と費用を見積もり直す（手順 16、`cost_check.py`）

3 節の CSV と 4 節の `metrics.csv` を `cost_check.py` に渡します。マシンは 1 台だけなので、区間は「このマシン」についてのもので、同じ GPU 名のほかのマシンには広げられません（`cost_check.py` がそう表示します）。

- データ待ちの比 η ＝ `train_toy.py` の処理量 ÷ ベンチの処理量。1 より小さい分は、GPU がデータ（CPU での合成）を待っている時間の目安です（III-5 で切り分けます）。ベンチと学習を同じ精度で測ったとき（CUDA なら `--amp`）だけ、こう読めます。
- 料金・為替・ディスク・存在した時間は仮の値です。Colab で動かした場合、実際の費用（残高の差）は Vast.ai のようには測れないので `--actual-jpy` は付けません。Vast.ai で借りたときは、教科書のとおり残高の差を円にして渡します。

In [ ]:
out5 = run(f"cost_check.py runs/bench_trace/{MACHINE}_audio_*.csv --total-steps 15640 "
           f"--price-usd-h {PRICE_USD_H} --usd-jpy {USD_JPY} --setup-min 20 --idle-min 15 "
           f"--disk-gb 40 --storage-usd-gb-month 0.15 --exist-h 0.9 "
           f"--estimate-jpy {EST[0]:.0f} {EST[1]:.0f} {EST[2]:.0f} --train-metrics {CK}/metrics.csv --out cost_check_{MACHINE}.png")
show(f"cost_check_{MACHINE}.png")
# 本番（教科書の手順16）：... --actual-jpy 70（残高の差を円にしたもの）を足す。値はすべて自分のものに

In [ ]:
# 確かめ（教科書 III-3-5 の「確かめること」と III-3-7 の結果の例のうち、機械に依らず成り立つことだけ）
ok("マシンが1台だけ" in out5, "マシンが 1 台なら「区間は『このマシン』についてのもの」と表示される（疑似反復を避ける）")
eta = num(r"サンプル/秒 = ([\d.]+)", out5)
s = mt.samples_per_s.to_numpy()[1:]
step_mean = np.mean([summarize_rep(g.sort_values("step").t_s.to_numpy())["mean"] for _, g in tr.groupby("rep")])
bench_rate = 128 / step_mean               # ベンチの処理量 [サンプル/秒]（MSER-5 で除いたあとの、実行ごとの平均の平均）
ok(abs(eta - np.exp(np.log(s).mean()) / bench_rate) < 0.011,
   f"η = {eta:.2f} は、最初のエポックを除いた処理量の幾何平均 ÷（128 ÷ 実行ごとの平均の平均）")
T = num(r"計算時間 ≈ ([\d.]+) 時間", out5)
ok(abs(T - 15640 * step_mean / 3600) < 0.006, f"本番の計算時間 {T:.2f} 時間 ＝ 15640 ステップ × 1 ステップの平均")
print(f"η = {eta:.2f}（III-3-7：train_toy.py の処理量は、ベンチの処理量より小さいことが多い。データを CPU で作るため）")

## 6. 発展課題 1：マシンの差の分散成分と、借りる台数

1 節の合成データ（3 台 × 3 回）の「実行の平均」$y_{ij}$ に、一元配置の変量効果モデル $y_{ij} = \mu + b_i + e_{ij}$（$b_i \sim N(0, \sigma_b^2)$、$e_{ij} \sim N(0, \sigma_r^2)$）を当てはめます。`cost_check.py` の分散分析（モーメント法）と、statsmodels の `mixedlm`（REML）で $\sigma_b$・$\sigma_r$ を推定して比べ、$\operatorname{Var}(\bar y) = \sigma_b^2/k + \sigma_r^2/(kr)$ から「平均の 95%区間を ±5% にするには何台借りればよいか」を計算します。

台数 $k$ を増やす代わりに同じマシンでのくり返し $r$ を増やしても、$\sigma_b^2/k$ の項は小さくならないことを確かめてください。

In [ ]:
import statsmodels.formula.api as smf
from scipy import stats
ym = rs.assign(y=rs["mean"] * 1e3)      # 1 節の「実行ごとの平均」[ms]
fit = smf.mixedlm("y ~ 1", ym, groups=ym["machine"]).fit(reml=True)
sb_reml, sr_reml = float(np.sqrt(fit.cov_re.iloc[0, 0])), float(np.sqrt(fit.scale))
sb_anova = num(r"マシン間の SD = ([\d.]+) ms", out1); sr_anova = num(r"実行間の SD = ([\d.]+) ms", out1)
print(f"σ_b：分散分析 {sb_anova:.2f} ms、REML {sb_reml:.2f} ms ／ σ_r：分散分析 {sr_anova:.2f} ms、REML {sr_reml:.2f} ms")
mu = fit.fe_params.iloc[0]


def half_width(k, r, sb, sr):
    return stats.t.ppf(0.975, k - 1) * np.sqrt(sb**2 / k + sr**2 / (k * r))


for r in (3, 10):
    need = next(k for k in range(2, 200) if half_width(k, r, sb_reml, sr_reml) <= 0.05 * mu)
    print(f"r = {r:2d} 回ずつ：±5%（±{0.05 * mu:.2f} ms）にするには {need} 台（k = 3 なら ±{half_width(3, r, sb_reml, sr_reml):.2f} ms）")

In [ ]:
# 確かめ（発展課題1）
ok(abs(sb_reml - sb_anova) < 0.1 * sb_anova and abs(sr_reml - sr_anova) < 0.15 * sr_anova,
   "釣り合った計画（3 台 × 3 回）では、REML と分散分析の推定はほぼ同じ")
ok(abs(half_width(3, 3, sb_anova, sr_anova) - half) < 0.05,
   f"k = 3、r = 3 の区間 ±{half_width(3, 3, sb_anova, sr_anova):.2f} ms は、1 節の cost_check.py の ±{half:.2f} ms と一致")
k3, k10 = (next(k for k in range(2, 200) if half_width(k, r, sb_reml, sr_reml) <= 0.05 * mu) for r in (3, 10))
ok(k10 >= k3 - 1, f"くり返しを 3 → 10 回にしても、必要な台数はほとんど減らない（{k3} → {k10} 台。σ_b²/k が大きいため）")

## 7. 結果を持ち帰る（手順 15 の `vast_sync.sh pull` の代わり）

Colab のファイルはランタイムを切ると消えるので、`runs/` を zip にしてダウンロードします（Mac で動かしているときは何もしません）。Mac では `~/home-cloud/runs/` の下に展開し、`uv run python cost_check.py runs/bench_trace/colab_audio_*.csv …` や `compare_runs.py`（III-2）で比べ、台帳（`cost_ledger.csv`）に 1 行書きます。

終わったら、Colab のランタイムを解放します（「ランタイム」メニューの「ランタイムを接続解除して削除」など。項目の名前は公式ドキュメントで要確認）。Vast.ai で同じことをしたときは、**回収を確かめてからインスタンスを削除**し、`vast_end_check.sh`（またはコンソール）で 0 台を確かめます。

In [ ]:
if IN_COLAB:
    zip_path = shutil.make_archive("p3c3_runs", "zip", root_dir=".", base_dir="runs")
    from google.colab import files
    files.download(zip_path)
    # from google.colab import runtime; runtime.unassign()   # 結果をダウンロードしてから。ランタイムを返す（III-2 のセル7）
else:
    print("Colab ではないので、持ち帰りはしません。結果は", os.path.abspath("runs"), "にあります")
print(sorted(os.listdir("runs")))
print(f"ok() の数：{N_OK}")

## おわりに

- 1 ステップの時間を全部残し、ウォームアップを MSER-5 で除き、自己相関を考えた区間（バッチ平均法）を出しました。独立な反復の単位は「マシン」で、1 台の中のステップや実行を別々の反復として数えると（疑似反復）、区間が何十倍も狭く出ます。
- 15640 ステップ程度の本番では、計算そのものより準備と待ち（$t_{\text{setup}} + t_{\text{idle}}$）が費用の大半になりやすいことを、見積もりの内訳で確かめました。
- SIGTERM で止めて同じコマンドで再開できることを確かめました。中断されるタイプのインスタンスや Colab の切断に備える形です。
- Vast.ai で実際に借りるときは、教科書の手順 1〜16 のとおりに、**止め方を先に確かめ、終了時刻のアラームを設定してから**作ってください。